In [0]:
# Databricks notebook source

from __future__ import annotations

import csv
import os
import uuid

from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StructType, StringType

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")

catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Resolve repo-relative path
# ============================================================
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = notebook_path.split("/notebooks/")[0]
workspace_csv_path = f"/Workspace{repo_root}/data/reference/txdot_item_specifications_classified.csv"

print("Workspace CSV path:", workspace_csv_path)

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "140_load_work_category_to_bronze.py"
RUN_ID = str(uuid.uuid4())

TARGET_TABLE = f"{catalog}.bronze.work_category_raw"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Validation
# ============================================================
if not os.path.exists(workspace_csv_path):
    raise RuntimeError(
        f"Missing reference file: {workspace_csv_path}\n"
        "Expected location: data/reference/txdot_item_specifications_classified.csv"
    )

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Ensure target table exists
# ============================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
      specification_description   STRING
    , item_work_category          STRING
)
USING DELTA
COMMENT 'Raw classified TxDOT item specification reference data.'
""")

spark.sql(f"""
COMMENT ON COLUMN {TARGET_TABLE}.specification_description
IS 'Specification description from the classified TxDOT item specification reference file.'
""")

spark.sql(f"""
COMMENT ON COLUMN {TARGET_TABLE}.item_work_category
IS 'Assigned work category from the classified TxDOT item specification reference file.'
""")

# ============================================================
# Read CSV with Python (driver-safe for repo file)
# ============================================================
rows: list[tuple[str, str]] = []

with open(workspace_csv_path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        specification_description = (row.get("specification_description") or "").strip()
        item_work_category = (row.get("work_category") or "").strip()

        if specification_description and item_work_category:
            rows.append((specification_description, item_work_category))

schema = StructType([
    StructField("specification_description", StringType(), True),
    StructField("item_work_category", StringType(), True),
])

df = spark.createDataFrame(rows, schema=schema).dropDuplicates(["specification_description"])

row_count = df.count()

# ============================================================
# Replace contents
# ============================================================
spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
df.write.mode("append").saveAsTable(TARGET_TABLE)

# ============================================================
# Log + summary
# ============================================================
log_run(
    status="SUCCESS",
    row_count=row_count,
    message=f"Built {TARGET_TABLE} successfully."
)

print("=" * 70)
print(f"Built {TARGET_TABLE}")
print(f"Row count: {row_count:,}")
print("=" * 70)